# 02. ETL: Поведенческие агрегаты

> **Краткое резюме**: Вычисляем поведенческие признаки для каждого клиента: обороты, количество транзакций, направленность потоков, динамика контрагентов, остатки, продукты. Результат — `data/behavioral_features.parquet`.

**Источники**:
- `paymentcounteragent_stran` — транзакции (основной)
- `clientbalance_sagg` — остатки на счетах
- `bmbpenetrproduct_stat` — продуктовое проникновение

**Предпосылки**: Запустите `01_etl_profile.ipynb` (создаёт вселенную клиентов).

---

In [ ]:
import os
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
log = logging.getLogger(__name__)

In [ ]:
# =====================================================
# ПАРАМЕТРЫ (должны совпадать с 01_etl_profile)
# =====================================================

HIVE_DATABASE = 's_dmrb'
START_DATE = '2025-07-01'
END_DATE   = '2025-12-31'

START_INT = int(START_DATE.replace('-', ''))
END_INT   = int(END_DATE.replace('-', ''))

# Середина периода для расчёта трендов (первая/вторая половина)
MID_DATE = '2025-10-01'
MID_INT  = int(MID_DATE.replace('-', ''))

# Pilot: процент клиентов для сэмплирования (10% — воспроизводимый hash-сэмпл)
# Для прода: SAMPLE_PCT = 100
SAMPLE_PCT = 10

OUTPUT_DIR = os.path.abspath('./data')
SPARK_OUTPUT_DIR = f'file://{OUTPUT_DIR}'

print(f'Period: {START_DATE} — {END_DATE}, mid: {MID_DATE}, sample: {SAMPLE_PCT}%')

In [ ]:
try:
    _ = spark.sparkContext.appName
    print(f'SparkSession available: {spark.sparkContext.appName}')
except Exception:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder \
        .appName('LookAlike_ETL_Behavior') \
        .enableHiveSupport() \
        .getOrCreate()
    print('SparkSession created')

spark.sql(f'USE {HIVE_DATABASE}')

---
## 1. Загрузка вселенной из 01_etl_profile

In [ ]:
profiles_path = f'{SPARK_OUTPUT_DIR}/client_profiles.parquet'
profiles_df = spark.read.parquet(profiles_path)
profiles_df.select('client_uk').createOrReplaceTempView('universe')
print(f'Universe loaded: {profiles_df.count():,} clients')

---
## 2. Транзакционные агрегаты (`paymentcounteragent_stran`)

Основной источник поведенческих фичей. Каждая транзакция записана с точки зрения `client_uk`:
- `income_flag = 'N'` → клиент платит (outflow)
- `income_flag = 'Y'` → клиент получает (inflow)

In [ ]:
# Оптимизации для большого объёма данных:
# 1. Убран JOIN с universe — universe создана из этой же таблицы с тем же фильтром
# 2. approx_count_distinct вместо COUNT(DISTINCT) — HLL-скетчи, ~2% погрешность
# 3. Результат на диск (.write.parquet) вместо .cache() — Spark spillит, не OOM
# 4. maxPartitionBytes=32m — мелкие блоки чтения, меньше давление на heap
# 5. Hash-сэмпл SAMPLE_PCT% клиентов — pilot не требует полного объёма

spark.conf.set('spark.sql.files.maxPartitionBytes', '32m')
spark.conf.set('spark.sql.shuffle.partitions', 1000)

sample_filter = f'AND ABS(HASH(p.client_uk)) % 100 < {SAMPLE_PCT}' if SAMPLE_PCT < 100 else ''

tx_agg_query = spark.sql(f"""
    SELECT
        p.client_uk,

        -- Объёмы
        COUNT(*)                                                    AS tx_count,
        SUM(p.rur_amt)                                              AS total_amount,
        SUM(CASE WHEN p.income_flag = 'N' THEN p.rur_amt ELSE 0 END)  AS total_outflow,
        SUM(CASE WHEN p.income_flag = 'Y' THEN p.rur_amt ELSE 0 END)  AS total_inflow,
        AVG(p.rur_amt)                                              AS avg_tx_amount,
        STDDEV(p.rur_amt)                                           AS std_tx_amount,

        -- Контрагенты (approx — HLL, ~2% погрешность, в разы легче по памяти)
        approx_count_distinct(p.client_contr_uk)                                              AS unique_counterparties,
        approx_count_distinct(CASE WHEN p.income_flag = 'N' THEN p.client_contr_uk END)       AS unique_payees,
        approx_count_distinct(CASE WHEN p.income_flag = 'Y' THEN p.client_contr_uk END)       AS unique_payers,

        -- Активность (макс 6 значений — точный COUNT DISTINCT ок)
        COUNT(DISTINCT CAST(FLOOR(p.date_part / 100) AS INT))       AS active_months,

        -- Тренды
        SUM(CASE WHEN p.date_part <  {MID_INT} THEN p.rur_amt ELSE 0 END)  AS amount_first_half,
        SUM(CASE WHEN p.date_part >= {MID_INT} THEN p.rur_amt ELSE 0 END)  AS amount_second_half,
        approx_count_distinct(CASE WHEN p.date_part <  {MID_INT} THEN p.client_contr_uk END)  AS cp_first_half,
        approx_count_distinct(CASE WHEN p.date_part >= {MID_INT} THEN p.client_contr_uk END)  AS cp_second_half

    FROM {HIVE_DATABASE}.paymentcounteragent_stran p
    WHERE p.date_part >= {START_INT} AND p.date_part <= {END_INT}
      AND (p.deleted_flag IS NULL OR p.deleted_flag != 'Y')
      AND p.rur_amt IS NOT NULL AND p.rur_amt > 0
      AND p.client_uk IS NOT NULL
      {sample_filter}
    GROUP BY p.client_uk
""")

tx_agg_path = f'{SPARK_OUTPUT_DIR}/tx_agg.parquet'
tx_agg_query.write.mode('overwrite').parquet(tx_agg_path)
print('Transaction aggregates written to disk')

tx_agg_df = spark.read.parquet(tx_agg_path)
tx_agg_df.createOrReplaceTempView('tx_universe')
print(f'Transaction aggregates: {tx_agg_df.count():,} clients')
tx_agg_df.describe('total_amount', 'tx_count', 'unique_counterparties', 'active_months').show()

---
## 3. Остатки на счетах (`clientbalance_sagg`)

Средний / макс / мин остаток за период. Даёт представление о «размере» клиента.

In [ ]:
# Верификация колонок (раскомментируйте при первом запуске)
# spark.sql(f'DESCRIBE {HIVE_DATABASE}.clientbalance_sagg').show(60, truncate=False)

In [ ]:
try:
    balance_df = spark.sql(f"""
        SELECT
            b.client_uk,
            AVG(b.balance_rur_amt)            AS avg_balance,
            MAX(b.balance_rur_amt)            AS max_balance,
            MIN(b.balance_rur_amt)            AS min_balance,
            AVG(b.balance_rur_amt_30_avg)     AS avg_balance_30d
        FROM {HIVE_DATABASE}.clientbalance_sagg b
        INNER JOIN tx_universe u ON b.client_uk = u.client_uk
        WHERE b.date_part >= {START_INT} AND b.date_part <= {END_INT}
        GROUP BY b.client_uk
    """)
    print(f'Balance aggregates: {balance_df.count():,} clients')
except Exception as e:
    print(f'Balance extraction failed: {e}')
    print('Check column names with DESCRIBE. Skipping.')
    balance_df = None

---
## 4. Продуктовое проникновение (`bmbpenetrproduct_stat`)

In [ ]:
try:
    product_df = spark.sql(f"""
        SELECT
            pp.client_uk,
            COUNT(DISTINCT pp.product_uk)           AS product_count,
            COUNT(DISTINCT pp.producttypemass_name)  AS product_type_count,
            SUM(pp.rur_amt)                          AS product_total_amt
        FROM {HIVE_DATABASE}.bmbpenetrproduct_stat pp
        INNER JOIN tx_universe u ON pp.client_uk = u.client_uk
        WHERE pp.date_part >= {START_INT} AND pp.date_part <= {END_INT}
        GROUP BY pp.client_uk
    """)
    print(f'Product aggregates: {product_df.count():,} clients')
except Exception as e:
    print(f'Product extraction failed: {e}')
    print('Skipping.')
    product_df = None

---
## 5. Сборка поведенческих фичей

In [ ]:
behavior_df = tx_agg_df

if balance_df is not None:
    behavior_df = behavior_df.join(balance_df, on='client_uk', how='left')

if product_df is not None:
    behavior_df = behavior_df.join(product_df, on='client_uk', how='left')

print(f'Behavioral features: {behavior_df.count():,} rows, {len(behavior_df.columns)} columns')
print(f'Columns: {behavior_df.columns}')

In [ ]:
# Сохранение
out_path = f'{SPARK_OUTPUT_DIR}/behavioral_features.parquet'
behavior_df.write.mode('overwrite').parquet(out_path)
print(f'Saved: {out_path}')

---
## 6. Статистика

In [ ]:
behavior_df.describe(
    'total_amount', 'total_outflow', 'total_inflow',
    'tx_count', 'unique_counterparties', 'active_months',
    'avg_tx_amount'
).show(truncate=False)

behavior_df.show(5, truncate=20, vertical=True)

---

## Глоссарий

| Термин | Описание |
|--------|----------|
| **total_outflow / total_inflow** | Сумма исходящих / входящих платежей за период |
| **unique_counterparties** | Количество уникальных контрагентов |
| **unique_payees / payers** | Уникальные получатели / плательщики |
| **active_months** | Количество месяцев с хотя бы 1 транзакцией |
| **amount_first_half / second_half** | Обороты в первой / второй половине периода (для тренда) |
| **cp_first_half / second_half** | Количество контрагентов в первой / второй половине |
| **avg_balance** | Средний остаток на счетах за период (RUR) |
| **product_count** | Количество уникальных банковских продуктов |

---

**Следующий шаг**: `03_feature_engineering.ipynb`.